# Lab 08 Challenge: Complete UniGPS Multi-Agent Support System

**Goal:** Build a production-grade multi-agent support system combining ALL patterns from this session.

**Scenario:**
UniGPS needs a complete support desk system that:
1. Uses an LLM supervisor to classify and route requests
2. Has specialized worker agents (HR, Tech, Finance, Facilities)
3. Supports handoff/escalation when a worker can't handle a request
4. Aggregates results for multi-domain requests
5. Implements fallback chains for reliability
6. Tracks everything in an audit trail
7. Uses checkpointing for conversation persistence

This exercise has LESS pre-written code — use what you learned in Labs 01-07!

Requires: `GROQ_API_KEY` in `.env`

## Setup: Imports and LLM

In [ ]:
import os
from typing import TypedDict, Annotated
from operator import add
from datetime import datetime
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

## State Definition (provided)

In [ ]:
class SupportRequest(TypedDict):
    # Input
    employee_name: str
    request: str
    # Routing
    category: str               # hr, tech, finance, facilities, general
    confidence: int             # 1-10 from LLM classifier
    # Processing
    worker_output: str
    needs_escalation: bool
    escalation_reason: str
    # Fallback
    error: str
    fallback_used: bool
    # Output
    final_response: str
    # Tracking
    audit: Annotated[list, add]

## Requirements

1. **LLM Supervisor (router)**
   - Classify request into: hr, tech, finance, facilities, general
   - Include confidence score (1-10)
   - If confidence < 5, route to "clarify" agent
   - Handle LLM errors gracefully (fallback to "general")

2. **Specialized Workers (4 domain agents)**
   - HR: leave, policies, insurance, onboarding
   - Tech: servers, laptops, deployments, bugs
   - Finance: expenses, salary, tax, reimbursement
   - Facilities: desk, parking, cafeteria, access cards
   - Each with domain-specific system prompt

3. **Escalation**
   - Worker can set `needs_escalation=True` if request is too complex
   - Escalation goes to a "manager_agent" node
   - Manager agent handles with broader authority

4. **Fallback Chain**
   - Primary (LLM specialist) -> Fallback (template) -> Error response
   - QA check: verify output length and quality

5. **Audit Trail**
   - Every node adds to audit: `Annotated[list, add]`
   - Include timestamps in audit entries

6. **Checkpointing**
   - MemorySaver with `thread_id` per employee
   - Support conversation history

**Graph structure:**
```
START -> supervisor -> [clarify | hr | tech | finance | facilities | general]
      -> [escalation_check] -> [manager | qa_check] -> [finalize | fallback]
      -> finalize -> END
```

## TODO 1: LLM Supervisor

Implement the supervisor node and routing function.

In [ ]:
# def supervisor(state: SupportRequest) -> dict:
#     """LLM-powered supervisor with confidence scoring."""
#     prompt = (
#         f"You are the UniGPS support desk supervisor.\n"
#         f"Classify this request into: hr, tech, finance, facilities, general\n"
#         f"Rate your confidence 1-10.\n"
#         f"Request: {state['request']}\n"
#         f"Reply:\nCATEGORY: ...\nCONFIDENCE: ..."
#     )
#     ...

# def route_supervisor(state: SupportRequest) -> str:
#     if state["confidence"] < 5:
#         return "clarify"
#     return state["category"]

## TODO 2: Specialized Workers

Implement the clarify agent and 4 domain workers + general worker.

In [ ]:
# def clarify_agent(state: SupportRequest) -> dict:
#     ...

# def hr_worker(state: SupportRequest) -> dict:
#     ...

# def tech_worker(state: SupportRequest) -> dict:
#     ...

# def finance_worker(state: SupportRequest) -> dict:
#     ...

# def facilities_worker(state: SupportRequest) -> dict:
#     ...

# def general_worker(state: SupportRequest) -> dict:
#     ...

## TODO 3: Escalation

Implement the escalation check, routing, and manager agent.

In [ ]:
# def escalation_check(state: SupportRequest) -> dict:
#     """Check if the worker flagged for escalation."""
#     ...

# def route_escalation(state: SupportRequest) -> str:
#     return "manager" if state["needs_escalation"] else "qa_check"

# def manager_agent(state: SupportRequest) -> dict:
#     """Manager agent with broader authority."""
#     ...

## TODO 4: QA Gate and Fallback

Implement QA validation, fallback templates, and routing.

In [ ]:
# def qa_check(state: SupportRequest) -> dict:
#     """Verify response quality."""
#     ...

# def route_qa(state: SupportRequest) -> str:
#     return "fallback" if state["error"] else "finalize"

# def fallback(state: SupportRequest) -> dict:
#     ...

## TODO 5: Finalize

Implement the finalize node that composes the final response.

In [ ]:
# def finalize(state: SupportRequest) -> dict:
#     timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
#     ...

## TODO 6: Build the Graph

Graph structure:
```
START -> supervisor -> [clarify | hr | tech | finance | facilities | general]
      -> escalation_check -> [manager | qa_check] -> [finalize | fallback]
      -> finalize -> END
```

In [ ]:
# Build graph
# graph = StateGraph(SupportRequest)
# ... add all nodes and edges ...

# memory = MemorySaver()
# app = graph.compile(checkpointer=memory)

## Test Cases

In [ ]:
test_requests = [
    {
        "employee_name": "Priya Sharma",
        "request": "I want to apply for 5 days casual leave from next Monday",
        "thread_id": "support-001",
    },
    {
        "employee_name": "Vikram Patel",
        "request": "The production database is running very slow and queries are timing out",
        "thread_id": "support-002",
    },
    {
        "employee_name": "Anita Desai",
        "request": "When will my travel expense reimbursement from last month be credited?",
        "thread_id": "support-003",
    },
    {
        "employee_name": "Rahul Kumar",
        "request": "I need a standing desk and a parking spot in the new building",
        "thread_id": "support-004",
    },
    {
        "employee_name": "Meera Joshi",
        "request": "asdfghjkl",
        "thread_id": "support-005",
    },
    {
        "employee_name": "Amit Singh",
        "request": "I need a policy exception for 30 days leave for my wedding",
        "thread_id": "support-006",
    },
]

In [ ]:
# TODO: Uncomment and run when your implementation is ready

# for req in test_requests:
#     config = {"configurable": {"thread_id": req["thread_id"]}}
#     print(f"Employee: {req['employee_name']}")
#     print(f"Request: {req['request']}")
#
#     result = app.invoke({
#         "employee_name": req["employee_name"],
#         "request": req["request"],
#         "category": "", "confidence": 0,
#         "worker_output": "", "needs_escalation": False,
#         "escalation_reason": "", "error": "",
#         "fallback_used": False, "final_response": "",
#         "audit": [],
#     }, config)
#
#     print(f"\n  Category: {result.get('category', 'N/A')}")
#     print(f"  Confidence: {result.get('confidence', 'N/A')}/10")
#     print(f"  Escalated: {result.get('needs_escalation', False)}")
#     print(f"  Fallback: {result.get('fallback_used', False)}")
#     print(f"  Response: {result.get('final_response', 'N/A')[:80]}...")
#     print(f"  Audit trail:")
#     for entry in result.get("audit", []):
#         print(f"    {entry}")
#     print()

In [ ]:
# TODO: Uncomment to see summary across all threads

# print(f"{'Thread':<14} {'Employee':<18} {'Category':<12} {'Escalated'}")
# for req in test_requests:
#     config = {"configurable": {"thread_id": req["thread_id"]}}
#     snap = app.get_state(config)
#     v = snap.values
#     print(
#         f"{req['thread_id']:<14} "
#         f"{req['employee_name']:<18} "
#         f"{v.get('category', 'N/A'):<12} "
#         f"{v.get('needs_escalation', False)}"
#     )

## Challenge Tips

Combine everything from Labs 01-07:
- **Lab 01:** LLM-powered supervisor routing
- **Lab 02:** Specialized worker agents with domain prompts
- **Lab 03:** Handoff and escalation patterns
- **Lab 04:** Result aggregation across agents
- **Lab 05:** Fallback chains for reliability
- **Lab 06:** Audit trail with `Annotated[list, add]` reducer
- **Lab 07:** Checkpointing with `MemorySaver` and `thread_id`

Patterns to implement:
- LLM supervisor with confidence routing (< 5 -> clarify)
- 5 specialized domain workers (HR, Tech, Finance, Facilities, General)
- Escalation path to manager agent for policy exceptions
- Fallback chain (LLM -> template on error/QA fail)
- QA gate validates response quality before delivery
- Audit trail with timestamps via `Annotated[list, add]`
- MemorySaver checkpointing with `thread_id` per employee

Check `solutions/lab08_challenge.ipynb` when you're done.